# Trajectory dataset

> Fixed-length windows of consecutive steps, for training world models and policies.

In [ ]:
#| default_exp data.dataset

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
from __future__ import annotations

from pathlib import Path
from typing import Any, Iterable

import numpy as np

from stable_marl.data.format import EpisodeReader, load_episodes

In [ ]:
#| export
TRANSITION_KEYS = ('action', 'reward', 'terminated', 'truncated')

def _is_step_key(key: str) -> bool:
    "Keys with one value per observation (T + 1 of them)."
    return key.startswith('obs.') or key == 'state'


class TrajectoryDataset:
    """
    Windows of `num_steps` consecutive observations from a collected dataset.

    Item ``i`` is a dict with, for one window starting at step ``t`` of an episode:

    * observations and states (``obs.<key>``, ``state``): steps ``t .. t + num_steps - 1``
    * transitions (``action``, ``reward``, ``terminated``, ``truncated``): the
      ``num_steps - 1`` transitions between those observations
    * per-episode values (``seed``, ``episode.mission``, ...) as is,
      plus ``episode`` (index in the dataset) and ``start`` (``t``)

    Items are dicts of numpy arrays, so a ``torch.utils.data.DataLoader`` can batch them.
    Episodes are cached in memory once read (``cache=False`` to re-read them every time);
    use `keys` to only load what you train on, e.g. without the rendered ``obs.pov``.
    """
    def __init__(
        self,
        path: str | Path | EpisodeReader,
        num_steps: int = 2,
        keys: Iterable[str] | None = None,
        format: str | None = None,
        cache: bool = True):
        assert num_steps >= 1
        self.episodes = path if isinstance(path, EpisodeReader) else load_episodes(path, format)
        self.num_steps, self.cache = num_steps, cache
        self.keys = None if keys is None else list(dict.fromkeys([*keys, 'length']))
        self._cache: dict[int, dict[str, np.ndarray]] = {}
        lengths = [int(self.episodes.episode(e, keys=['length'])['length'])
                   for e in range(self.episodes.num_episodes)]
        # a window needs num_steps observations: episodes of length T have T + 1 of them
        self._index = np.array([(e, t) for e, T in enumerate(lengths) for t in range(T + 2 - num_steps)],
                               dtype=np.int64).reshape(-1, 2)

    @property
    def metadata(self) -> dict[str, Any]:
        return self.episodes.metadata

    def __len__(self) -> int:
        return len(self._index)

    def episode(self, index: int) -> dict[str, np.ndarray]:
        "The (cached) arrays of episode `index`."
        if index in self._cache:
            return self._cache[index]
        episode = self.episodes.episode(index, keys=self.keys)
        if self.cache:
            self._cache[index] = episode
        return episode

    def __getitem__(self, i: int) -> dict[str, np.ndarray]:
        e, t = (int(x) for x in self._index[i])
        n, item = self.num_steps, {}
        for key, value in self.episode(e).items():
            if _is_step_key(key):
                item[key] = value[t:t + n]
            elif key in TRANSITION_KEYS:
                item[key] = value[t:t + n - 1]
            else:
                item[key] = value
        item['episode'], item['start'] = np.array(e), np.array(t)
        return item

### Tests

In [ ]:
import tempfile
from stable_marl.data.npz import NpzWriter

def episode(T):
    return {'obs.image': np.arange(T + 1)[:, None, None].repeat(2, 1), 'state': np.arange(T + 1),
            'action': np.arange(T)[:, None].repeat(2, 1), 'reward': np.zeros((T, 2)), 'length': np.array(T)}

with tempfile.TemporaryDirectory() as tmp:
    with NpzWriter(tmp) as w: w.write_episodes([episode(4), episode(1)])
    ds = TrajectoryDataset(tmp, num_steps=3)
    assert len(ds) == 3 + 0     # T=4: 5 obs -> 3 windows of 3; T=1: 2 obs -> none
    item = ds[2]
    assert list(item['state']) == [2, 3, 4] and item['action'].shape == (2, 2) and list(item['action'][:, 0]) == [2, 3]
    assert item['obs.image'].shape == (3, 2, 1) and int(item['start']) == 2
    assert len(TrajectoryDataset(tmp, num_steps=1)) == 5 + 2
    assert set(TrajectoryDataset(tmp, num_steps=2, keys=['state'])[0]) == {'state', 'length', 'episode', 'start'}

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()